In [ ]:
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from IPython.display import display
import matplotlib.pyplot as plt

OUT = Path('.').resolve() / 'results_gpu'
df = pd.concat([pd.read_csv(f) for f in sorted(OUT.glob('stage2_*.csv'))])

# Filtrar solo moléculas válidas (sin alerts, SMILES válido)
df_valid = df[
    (df['Alerts'] == 1.0) &
    (df['SMILES_state'] == 1) &
    (df['Score'] > 0)
].copy()

print(f"Total moléculas generadas : {len(df):,}")
print(f"Válidas (sin alerts)      : {len(df_valid):,}")
print(f"\nTop 5 por ChemProp_5HT2A:")
print(df_valid.sort_values('ChemProp_5HT2A', ascending=False)
      [['SMILES','ChemProp_5HT2A','QED','TPSA (raw)','pActivity_pred (ChemProp_5HT2A)','step']]
      .head(5).to_string(index=False))

In [ ]:
# Visualizar top 12 moléculas
top12 = df_valid.sort_values('ChemProp_5HT2A', ascending=False).head(12)

mols = []
legends = []
for _, row in top12.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol:
        mols.append(mol)
        legends.append(
            f"pAct={row['pActivity_pred (ChemProp_5HT2A)']:.2f}\n"
            f"QED={row['QED']:.2f}"
        )

img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(350, 250), legends=legends)
display(img)

## Análisis de las moléculas generadas: distribución de las métricas simuladas y una visualización en 2D de las mejores moléculas para la diana 5-HT2A.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df_valid['ChemProp_5HT2A'], bins=20, ax=axes[0], color='skyblue')
axes[0].set_title('Distribución de ChemProp_5HT2A')

sns.histplot(df_valid['QED'], bins=20, ax=axes[1], color='lightgreen')
axes[1].set_title('Distribución de QED')

sns.histplot(df_valid['TPSA (raw)'], bins=20, ax=axes[2], color='salmon')
axes[2].set_title('Distribución de TPSA')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df_valid, x='QED', y='ChemProp_5HT2A', alpha=0.7, color='purple')
plt.title('ChemProp_5HT2A score vs QED')
plt.show()


In [ ]:
# Dibujar las 9 mejores moléculas
top_mols_df = df_valid.sort_values('ChemProp_5HT2A', ascending=False).head(9)
mols = [Chem.MolFromSmiles(smi) for smi in top_mols_df['SMILES']]
legends = [f"Chem:{row['ChemProp_5HT2A']:.3f} | QED:{row['QED']:.2f}" for _, row in top_mols_df.iterrows()]

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=legends)
display(img)
